# DWM Lab 8 — Agglomerative Hierarchical Clustering

### Dataset: Online Retail

**Objective:**  
To perform **customer segmentation using Agglomerative Hierarchical Clustering** and compare different linkage methods.

The experiment includes:

- Customer-level feature engineering
- Feature scaling
- Sampling for efficient hierarchical clustering
- Single Linkage
- Complete Linkage
- Average Linkage
- Ward Linkage
- Dendrogram visualization
- Silhouette Score comparison
- Cluster analysis

### Dataset adaptation

The same **Online Retail.xlsx** dataset is used for this experiment and for Lab 7.

Customer-level behavioural features are created from transaction data:

- **TotalSpending**
- **TotalOrders**
- **TotalQuantity**
- **AverageOrderValue**
- **UniqueProducts**

These features are standardized before clustering.

## 1. Import Required Libraries

Pandas and NumPy are used for data preparation. Matplotlib is used for visualization. Scikit-learn provides scaling, hierarchical clustering and silhouette evaluation. SciPy is used to generate dendrograms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

## 2. Load the Dataset

Load the same `Online Retail.xlsx` dataset used in the previous DWM experiments.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Preprocessing

For customer segmentation:

1. Convert `InvoiceDate` to datetime.
2. Remove cancelled invoices.
3. Remove transactions with non-positive quantity or price.
4. Calculate `SalesAmount`.
5. Remove records without `CustomerID`.
6. Convert `CustomerID` into an integer representation.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid sales transactions
df = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()

# Calculate sales amount
df['SalesAmount'] = df['Quantity'] * df['UnitPrice']

# Customer ID is required for customer-level clustering
df = df.dropna(subset=['CustomerID']).copy()
df['CustomerID'] = df['CustomerID'].astype(int)

print("Cleaned transaction shape:", df.shape)
display(df.head())

## 4. Create Customer-Level Features

Hierarchical clustering is performed on customers rather than individual transactions.

For every customer, calculate:

- Total Spending
- Total Orders
- Total Quantity
- Average Order Value
- Number of Unique Products

In [ ]:
customer_df = df.groupby('CustomerID').agg(
    TotalSpending=('SalesAmount', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    UniqueProducts=('StockCode', 'nunique')
).reset_index()

# Average order value
customer_df['AverageOrderValue'] = (
    customer_df['TotalSpending'] / customer_df['TotalOrders']
)

features = [
    'TotalSpending',
    'TotalOrders',
    'TotalQuantity',
    'AverageOrderValue',
    'UniqueProducts'
]

print("Number of customers:", len(customer_df))
display(customer_df.head())

## 5. Prepare Features

Retail customer features can be highly skewed because a small number of customers may have very high spending or order quantities.

A `log1p` transformation is applied before standardization.

In [ ]:
X_features = customer_df[features].copy()

# Log transformation
X_log = np.log1p(X_features)

display(X_log.head())

## 6. Feature Scaling

Agglomerative clustering is distance-based, so features must be placed on comparable scales.

`StandardScaler` transforms the features to approximately zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("Scaled feature matrix shape:", X_scaled.shape)

## 7. Sample Customers for Hierarchical Clustering

Hierarchical clustering and dendrogram construction can become computationally expensive for a very large number of observations.

Therefore, a random sample of **5,000 customers** is used for this experiment, following the same practical approach as the lab.

If fewer than 5,000 customers are available, all customers are used.

In [ ]:
sample_size = min(5000, len(customer_df))

rng = np.random.RandomState(42)
sample_indices = rng.choice(
    len(customer_df),
    size=sample_size,
    replace=False
)

X_sample = X_scaled[sample_indices]
customer_sample = customer_df.iloc[sample_indices].copy().reset_index(drop=True)

print("Customers used for hierarchical clustering:", len(customer_sample))

## 8. Hierarchical Clustering and Linkage

Agglomerative Hierarchical Clustering starts with every observation as its own cluster and repeatedly merges the closest clusters.

Four common linkage methods are compared:

### Single Linkage
Distance between two clusters is based on the closest pair of observations.

### Complete Linkage
Distance is based on the farthest pair of observations.

### Average Linkage
Distance is based on the average pairwise distance between clusters.

### Ward Linkage
Merges clusters while minimizing the increase in within-cluster variance.

## 9. Dendrogram — Single Linkage

A dendrogram shows the sequence of cluster merges and their distances.

In [ ]:
Z_single = linkage(X_sample, method='single', metric='euclidean')

plt.figure(figsize=(14, 6))
dendrogram(
    Z_single,
    truncate_mode='lastp',
    p=30
)
plt.title('Hierarchical Clustering Dendrogram — Single Linkage')
plt.xlabel('Customer Groups')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 10. Dendrogram — Complete Linkage

In [ ]:
Z_complete = linkage(X_sample, method='complete', metric='euclidean')

plt.figure(figsize=(14, 6))
dendrogram(
    Z_complete,
    truncate_mode='lastp',
    p=30
)
plt.title('Hierarchical Clustering Dendrogram — Complete Linkage')
plt.xlabel('Customer Groups')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 11. Dendrogram — Average Linkage

In [ ]:
Z_average = linkage(X_sample, method='average', metric='euclidean')

plt.figure(figsize=(14, 6))
dendrogram(
    Z_average,
    truncate_mode='lastp',
    p=30
)
plt.title('Hierarchical Clustering Dendrogram — Average Linkage')
plt.xlabel('Customer Groups')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 12. Dendrogram — Ward Linkage

In [ ]:
Z_ward = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(
    Z_ward,
    truncate_mode='lastp',
    p=30
)
plt.title('Hierarchical Clustering Dendrogram — Ward Linkage')
plt.xlabel('Customer Groups')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 13. Apply Agglomerative Clustering

For comparison, four models are created with **4 clusters**:

- Single
- Complete
- Average
- Ward

The same sampled customer feature matrix is used for all four models.

In [ ]:
n_clusters = 4

models = {
    'Single': AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage='single'
    ),
    'Complete': AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage='complete'
    ),
    'Average': AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage='average'
    ),
    'Ward': AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage='ward'
    )
}

cluster_labels = {}

for name, model in models.items():
    labels = model.fit_predict(X_sample)
    cluster_labels[name] = labels

print("Clustering completed for all linkage methods.")

## 14. Calculate Silhouette Scores

The Silhouette Score evaluates how well each customer fits within its assigned cluster.

A score closer to **1** generally indicates better separation, while values around zero indicate overlapping clusters.

In [ ]:
silhouette_results = []

for name, labels in cluster_labels.items():
    # Silhouette requires at least two distinct clusters
    if len(np.unique(labels)) > 1:
        score = silhouette_score(X_sample, labels)
    else:
        score = np.nan

    silhouette_results.append({
        'Linkage': name,
        'Silhouette Score': score
    })

silhouette_df = pd.DataFrame(silhouette_results)

display(silhouette_df.sort_values(
    'Silhouette Score',
    ascending=False
).reset_index(drop=True))

## 15. Visualize Silhouette Score Comparison

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    silhouette_df['Linkage'],
    silhouette_df['Silhouette Score']
)
plt.title('Silhouette Score by Linkage Method')
plt.xlabel('Linkage Method')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

## 16. Cluster Sizes for Each Linkage Method

Check how many customers are assigned to each cluster.

In [ ]:
cluster_size_tables = {}

for name, labels in cluster_labels.items():
    counts = pd.Series(labels).value_counts().sort_index()
    cluster_size_tables[name] = counts

    print(f"\n{name} Linkage:")
    print(counts)

## 17. Analyze the Clusters

For each linkage method, calculate the average customer characteristics inside each cluster.

In [ ]:
for name, labels in cluster_labels.items():
    analysis_df = customer_sample.copy()
    analysis_df['Cluster'] = labels

    summary = analysis_df.groupby('Cluster')[features].mean().round(2)

    print(f"\n{'=' * 60}")
    print(f"{name} Linkage — Cluster Summary")
    print('=' * 60)
    display(summary)

## 18. Visualize One Clustering Result

The Ward linkage result is visualized using total spending and total orders.

Ward linkage is used here only as a visualization example; the silhouette scores above should be considered when comparing the linkage methods.

In [ ]:
ward_df = customer_sample.copy()
ward_df['Cluster'] = cluster_labels['Ward']

plt.figure(figsize=(10, 6))

for cluster in sorted(ward_df['Cluster'].unique()):
    subset = ward_df[ward_df['Cluster'] == cluster]
    plt.scatter(
        subset['TotalOrders'],
        subset['TotalSpending'],
        label=f'Cluster {cluster}',
        alpha=0.6
    )

plt.title('Ward Hierarchical Clustering — Customer Segments')
plt.xlabel('Total Orders')
plt.ylabel('Total Spending')
plt.legend()
plt.tight_layout()
plt.show()

## 19. Final Comparison

Create a final table containing the silhouette score and number of clusters produced by each linkage method.

In [ ]:
final_comparison = silhouette_df.copy()
final_comparison['Number of Clusters'] = [
    len(np.unique(cluster_labels[name]))
    for name in final_comparison['Linkage']
]

display(final_comparison)

## Conclusion

Agglomerative Hierarchical Clustering was successfully performed on the Online Retail dataset.

The experiment included:

- Customer-level feature engineering
- Log transformation
- Standardization
- Sampling 5,000 customers for computational efficiency
- Dendrograms for Single, Complete, Average and Ward linkage
- Agglomerative clustering with 4 clusters
- Silhouette Score comparison
- Cluster-size analysis
- Cluster-wise customer analysis
- Customer segmentation visualization

The same customer-level features can be reused from Lab 7, allowing both **K-Means** and **Hierarchical Clustering** to be performed on the same dataset.